# Figure 3O: MRGPRX4/PRRX1 cells relative to CD31+ vessels and PROX1+ lymphatics

Gour et al. 2026, eLife (Reviewed Preprint 112036).

This notebook performs the quantification underlying Figure 3O. It segments nuclei
from Hoechst, defines the BV/LV+ neighbourhood from CD31 and PROX1, scores each
nucleus as MRGPRX4+ and/or PRRX1+, and writes two tables:

- `bv_lv_X4_PRRX1_per_nucleus_allfields.csv`
- `bv_lv_X4_PRRX1_per_field_percentages.csv`

The plot and statistics shown in Figure 3O were generated separately from
`bv_lv_X4_PRRX1_per_field_percentages.csv` (see Methods).

**Input:** single-channel 2D TIFFs in one folder, named
`<SampleID>_<regionID>_<Marker>.tif`, with Marker one of
`Hoechst`, `X4`, `PRRX1`, `CD31`, `PROX1`.


In [ ]:
import os, glob, numpy as np, pandas as pd
from skimage import io, filters, morphology, measure, segmentation, exposure, util
from skimage.filters import threshold_otsu, sato, meijering
from skimage.morphology import remove_small_objects, remove_small_holes, disk
from scipy import ndimage as ndi

# ---- user inputs ----
image_folder = r""  # set to the folder holding all TIFFs, named
                    # <SampleID>_<regionID>_<Marker>.tif (e.g. NGMEL4_region2_Hoechst.tif)
pixel_size_um = 0.568182
vessel_neighborhood_um = 5.0
min_nucleus_area_um2 = 25
nuclear_dilate_um = 2.0
min_vessel_object_um2 = 10
use_vesselness = True
vesselness_method = "sato"

x4_threshold_mode = "otsu"
x4_percentile = 99.0
prrx1_threshold_mode = "otsu"
prrx1_percentile = 99.0

per_field_csv = "bv_lv_X4_PRRX1_per_field_percentages.csv"
per_nucleus_csv = "bv_lv_X4_PRRX1_per_nucleus_allfields.csv"

required_markers = ["Hoechst", "X4", "PRRX1", "CD31", "PROX1"]

def um2_to_px2(area_um2, pix_um):
    return max(1, int(round(area_um2 / (pix_um**2))))

def um_to_px(dist_um, pix_um):
    return max(1, int(round(dist_um / pix_um)))

def find_samples_regions_from_filenames(folder):
    """
    Expect filenames like:
      SampleID_regionX_Hoechst.tif
      SampleID_regionX_X4.tif
    Parses:
      sample_id = first chunk
      region_id = second chunk
      marker    = last chunk
    and builds:
      {sample_id: {region_id: {marker: path, ...}, ...}, ...}
    """
    samples = {}
    for ext in ("*.tif", "*.tiff"):
        for path in glob.glob(os.path.join(folder, ext)):
            base = os.path.splitext(os.path.basename(path))[0]
            parts = base.split("_")
            if len(parts) < 3:
                # not matching pattern, skip
                continue
            sample_id = parts[0]         # e.g. NGMEL4
            region_id = parts[1]         # e.g. region2
            marker    = parts[-1]        # e.g. Hoechst

            if marker not in required_markers:
                continue

            samples.setdefault(sample_id, {})
            samples[sample_id].setdefault(region_id, {})
            samples[sample_id][region_id][marker] = path

    # drop incomplete regions
    complete = {}
    for sample_id, regions in samples.items():
        good_regions = {
            r: d for r, d in regions.items()
            if all(m in d for m in required_markers)
        }
        if good_regions:
            complete[sample_id] = good_regions

    return complete

samples = find_samples_regions_from_filenames(image_folder)

print("Found samples and regions:")
for s, regs in samples.items():
    print(f"  {s}: {sorted(regs.keys())}")
def load_region_channels(paths_dict):
    chans = {}
    for m in required_markers:
        img = io.imread(paths_dict[m])
        img = util.img_as_float32(img)
        if img.ndim != 2:
            raise ValueError(f"{os.path.basename(paths_dict[m])} is not 2D (shape={img.shape})")
        chans[m] = img
    H, W = chans["Hoechst"].shape
    return chans, (H, W)

def segment_nuclei(hoechst, pix_um, min_area_um2, dilate_um):
    ho = filters.gaussian(hoechst, sigma=1)
    ho = exposure.rescale_intensity(ho, in_range="image", out_range=(0,1))
    th = threshold_otsu(ho)
    nuc = ho > th
    nuc = remove_small_objects(nuc, um2_to_px2(min_area_um2, pix_um))
    nuc = morphology.binary_closing(nuc, footprint=disk(1))
    dist = ndi.distance_transform_edt(nuc)
    maxima = morphology.h_maxima(dist, 0.5)
    markers = measure.label(maxima)
    nuc_labels = segmentation.watershed(-dist, markers, mask=nuc)
    nuc_labels_dil = segmentation.expand_labels(nuc_labels, distance=um_to_px(dilate_um, pix_um))
    return nuc_labels, nuc_labels_dil

def segment_vessel_like(channel_img, pix_um, min_obj_um2, use_vn=True, method="sato"):
    im = filters.gaussian(channel_img, sigma=1)
    if use_vn:
        vn = sato(im) if method == "sato" else meijering(im)
        vn = exposure.rescale_intensity(vn, in_range="image", out_range=(0,1))
        t = threshold_otsu(vn)
        m = vn > t
    else:
        t = threshold_otsu(im)
        m = im > t
    m = remove_small_objects(m, um2_to_px2(min_obj_um2, pix_um))
    m = remove_small_holes(m, area_threshold=um2_to_px2(5, pix_um))
    m = morphology.binary_closing(m, footprint=disk(1))
    return m

def per_nucleus_intensity(labels, img):
    props = measure.regionprops_table(
        labels, intensity_image=img,
        properties=("label", "mean_intensity", "centroid")
    )
    return pd.DataFrame(props)

def find_thr(vals, mode="otsu", pct=99.0):
    v = np.asarray(vals, dtype=np.float32)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return 0.0
    if mode == "otsu":
        return float(threshold_otsu(v) if (v > 0).sum() >= 5 else np.percentile(v, 95))
    return float(np.percentile(v, pct))

def classify_and_summarize(sample_id, region_id, paths_dict):
    chans, (H, W) = load_region_channels(paths_dict)
    Hoechst = chans["Hoechst"]
    X4      = chans["X4"]
    PRRX1   = chans["PRRX1"]
    CD31    = chans["CD31"]
    PROX1   = chans["PROX1"]

    # nuclei
    nuc_labels, nuc_labels_dil = segment_nuclei(
        Hoechst, pixel_size_um, min_nucleus_area_um2, nuclear_dilate_um
    )

    # vessels + lymphatics
    cd31_mask  = segment_vessel_like(CD31, pixel_size_um, min_vessel_object_um2,
                                     use_vesselness, vesselness_method)
    prox1_mask = segment_vessel_like(PROX1, pixel_size_um, min_vessel_object_um2,
                                     use_vesselness, vesselness_method)
    vessel_mask = cd31_mask | prox1_mask
    vessel_plus = ndi.binary_dilation(
        vessel_mask, structure=disk(um_to_px(vessel_neighborhood_um, pixel_size_um))
    )

    # per-nucleus signal
    df_x4  = per_nucleus_intensity(nuc_labels_dil, X4)
    df_prr = per_nucleus_intensity(nuc_labels_dil, PRRX1)
    df = df_x4.merge(df_prr[["label", "mean_intensity"]],
                     on="label", suffixes=("_x4", "_prrx1"))

    cy = df["centroid-0"].round().clip(0, H-1).astype(int)
    cx = df["centroid-1"].round().clip(0, W-1).astype(int)
    df["region_type"] = np.where(vessel_plus[cy, cx], "BV/LV+", "BV/LV-")

    thr_x4   = find_thr(df["mean_intensity_x4"],   mode=x4_threshold_mode,   pct=x4_percentile)
    thr_prrx = find_thr(df["mean_intensity_prrx1"], mode=prrx1_threshold_mode, pct=prrx1_percentile)

    df["X4+"]        = df["mean_intensity_x4"]    >= thr_x4
    df["PRRX1+"]     = df["mean_intensity_prrx1"] >= thr_prrx
    df["X4+PRRX1+"]  = df["X4+"] & df["PRRX1+"]
    df["Negative"]   = ~(df["X4+"] | df["PRRX1+"])

    # tag with sample + region
    df["sample"] = sample_id
    df["region"] = region_id

    # per-field %
    out = []
    for rt in ["BV/LV+", "BV/LV-"]:
        sub = df[df["region_type"] == rt]
        n = len(sub)
        pct = {"Negative": np.nan, "X4+": np.nan, "PRRX1+": np.nan, "X4+PRRX1+": np.nan}
        if n:
            pct = {
                "Negative":   100.0 * sub["Negative"].mean(),
                "X4+":        100.0 * sub["X4+"].mean(),
                "PRRX1+":     100.0 * sub["PRRX1+"].mean(),
                "X4+PRRX1+":  100.0 * sub["X4+PRRX1+"].mean()
            }
        pct["sample"]       = sample_id
        pct["region"]       = region_id
        pct["region_type"]  = rt
        pct["nuclei_total"] = n
        out.append(pct)

    return df, pd.DataFrame(out)
all_nuclei, all_fields = [], []

for sample_id, regions in samples.items():
    for region_id, paths_dict in regions.items():
        print(f"Analyzing {sample_id} / {region_id} ...")
        df_nuc, df_field = classify_and_summarize(sample_id, region_id, paths_dict)
        all_nuclei.append(df_nuc)
        all_fields.append(df_field)

df_nuclei = pd.concat(all_nuclei, ignore_index=True)
df_fields = pd.concat(all_fields,  ignore_index=True)

df_nuclei.to_csv(per_nucleus_csv, index=False)
df_fields.to_csv(per_field_csv, index=False)

print("Per-nucleus table:", per_nucleus_csv)
print("Per-field table:",  per_field_csv)
